# PKLot YOLO Training auf Google Colab
## GroupSplittet nach Aufnahmetag (0% Data Leakage)

Dieses Notebook trainiert YOLOv8 auf dem PKLot-Datensatz mit GroupShuffleSplit-Aufteilung.

**Wichtig:** Der Datensatz `dataset_grouped_yolo` muss bereits in Google Drive sein unter:
```
My Drive/dataset_grouped_yolo/
├── train/
├── val/
├── test/
└── data.yaml
```

## Setup: Dependencies installieren

In [ ]:
# YOLOv8 und Dependenzen installieren
!pip install -q ultralytics opencv-python pillow

print("Dependencies installiert")

## Google Drive mounten

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Verifiziere dass dataset_grouped_yolo existiert
dataset_path = '/content/drive/MyDrive/dataset_grouped_yolo'
if os.path.exists(dataset_path):
    print(f"Datensatz gefunden: {dataset_path}")
    
    # Zeige Struktur
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(dataset_path, split)
        if os.path.exists(split_path):
            n_images = len(os.listdir(os.path.join(split_path, 'images')))
            n_labels = len(os.listdir(os.path.join(split_path, 'labels')))
            print(f"  {split}: {n_images} Bilder, {n_labels} Labels")
else:
    print(f" Datensatz nicht gefunden: {dataset_path}")
    print("Bitte stelle sicher, dass dataset_grouped_yolo/ auf Google Drive ist")

## Datensatz-Validierung

In [ ]:
import yaml
from pathlib import Path

# Lade data.yaml
data_yaml_path = '/content/drive/MyDrive/dataset_grouped_yolo/data.yaml'

with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print(" Datensatz-Konfiguration:")
print(f"  Klassen: {data_config['nc']}")
print(f"  Klassen-Namen: {data_config['names']}")
print(f"  Train-Pfad: {data_config['path']}/{data_config['train']}")
print(f"  Val-Pfad: {data_config['path']}/{data_config['val']}")
print(f"  Test-Pfad: {data_config['path']}/{data_config['test']}")

print("\nDatensatz validiert!")

## YOLOv8 Modell laden & Parameter setzen

In [ ]:
from ultralytics import YOLO

# Lade ein vortrainiertes YOLOv8-Modell
# Optionen: yolov8n (nano), yolov8s (small), yolov8m (medium), yolov8l (large), yolov8x (xlarge)

model = YOLO('yolov8m.pt')

print(" YOLOv8m Modell geladen")
print(f"   Modell-Größe: medium")
print(f"   Vortraining: COCO Dataset")

## TRAINING STARTEN 

**Trainings-Parameter:**
- **epochs**: 100 (Anzahl Durchläufe über Datensatz)
- **imgsz**: 640 (Eingabebild-Größe)
- **batch**: 16 (Bilder pro Batch, angepasst für Colab GPU)
- **patience**: 20 (Early stopping bei keiner Verbesserung)
- **device**: 0 (GPU, standard in Colab)

Das Training dauert ca. 30-60 Minuten je nach GPU.

In [ ]:
# Trainiere das Modell
results = model.train(
    data='/content/drive/MyDrive/dataset_grouped_yolo/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project='/content/drive/MyDrive',
    name='pklot_yolo_training',
    exist_ok=False,
    verbose=True
)

print("\n🎉 Training abgeschlossen!")

## Modell-Evaluation auf Test-Set

In [ ]:
# Evaluiere auf Test-Set
results = model.val(
    data='/content/drive/MyDrive/dataset_grouped_yolo/data.yaml',
    split='test'  # Evaluiere auf Test-Bildern
)

print("\nTest-Evaluation abgeschlossen!")

## Inferenz: Vorhersagen auf einzelnen Bildern testen

In [ ]:
import cv2
from matplotlib import pyplot as plt
import os
from pathlib import Path

# Wähle ein Test-Bild
test_images_dir = '/content/drive/MyDrive/dataset_grouped_yolo/test/images'
test_images = os.listdir(test_images_dir)[:3]  # Erste 3 Bilder

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, image_name in enumerate(test_images):
    image_path = os.path.join(test_images_dir, image_name)
    
    # Mache Vorhersage
    results = model.predict(image_path, conf=0.25)
    
    # Zeige Ergebnis
    result_img = results[0].plot()  # Zeichne BBoxes
    
    axes[idx].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(f'Vorhersage: {image_name[:30]}')
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/pklot_predictions.jpg', dpi=100, bbox_inches='tight')
plt.show()

print("Vorhersagen gespeichert: /content/drive/MyDrive/pklot_predictions.jpg")

## Trainiertes Modell exportieren

In [ ]:
# Exportiere das trainierte Modell
exported_model_path = model.export(format='torchscript')  # oder 'onnx', 'tflite', etc.

print(f" Modell exportiert: {exported_model_path}")
print(f"\nSpeicherort: /content/drive/MyDrive/pklot_yolo_training/weights/best.pt")

## Zusammenfassung

In [ ]:
print("\n" + "="*70)
print("TRAINING ZUSAMMENFASSUNG")
print("="*70)
print(f"""
Datensatz: PKLot (GroupShuffleSplit nach Aufnahmetag)
   - Train: 8706 Bilder / 69 Tage
   - Val: 2087 Bilder / 16 Tage  
   - Test: 1623 Bilder / 15 Tage
   - Garantie: 0% Tages-Überlap (kein Data Leakage!)

Modell: YOLOv8m (medium)
   - Vortraining: COCO Dataset
   - Klassen: 3 (spaces, space-empty, space-occupied)

Ergebnisse speichert in:
   /content/drive/MyDrive/pklot_yolo_training/
   ├── weights/best.pt (bestes Modell)
   ├── results.csv (Metriken)
   └── predictions/ (Visualisierungen)

""")
print("="*70)